# SnapChef - YOLOv8 Edge Vision Training
This notebook is designed to be run in Google Colab to train a lightweight YOLOv8 Nano model on a food ingredients dataset, and export it to an `int8` quantized `.tflite` format for use in React Native.

## 1. Setup Environment

In [ ]:
!pip install -q ultralytics roboflow

## 2. Download Dataset from Roboflow
1. Go to Roboflow Universe and find a food ingredients dataset (YOLOv8 format).
2. Click 'Download Dataset' -> 'Show Download Code'.
3. Paste your Roboflow API key and workspace/project details below.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="mHMwqsFzwVtHmAeu2bTg")
project = rf.workspace("food-recipe-ingredient-images-0gnku").project("food-ingredients-dataset")
version = project.version(4)
dataset = version.download("yolov8")

## 3. Train YOLOv8 Nano

In [ ]:
from ultralytics import YOLO

# Load the lightweight nano model
model = YOLO('yolov8n.pt')

# Train the model. We use imgsz=320 for faster mobile inference.
# Note: dataset.location contains the path to the downloaded data.yaml
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=320,
    batch=16,
    project="snapchef_vision",
    name="edge_model"
)

## 4. Verify & Test the Model
Let's run the trained model on a test image to make sure it's working perfectly.

In [ ]:
import glob
from IPython.display import Image, display

# Grab a random image from the test set
test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")
if len(test_images) > 0:
    test_image = test_images[0]
    
    # Run inference
    res = model.predict(source=test_image, imgsz=320, save=True, project="snapchef_vision", name="test_inference")
    
    # Display the result
    display(Image(filename=f"snapchef_vision/test_inference/{test_image.split('/')[-1]}"))
else:
    print("No test images found.")

## 5. Export to TFLite (INT8 Quantized)
Export the model to run on edge devices using TensorFlow Lite. INT8 quantization shrinks the model size and makes it much faster on mobile CPUs/NPUs.

In [ ]:
# Export to TFLite
model.export(
    format='tflite',
    int8=True,       # Quantize to int8 for maximum speed
    imgsz=320,
    data=f"{dataset.location}/data.yaml" # Required for accurate int8 calibration
)

## 6. Download the Model
Run this to download your `.tflite` model directly to your computer. You will place this file in your React Native app!

In [ ]:
from google.colab import files

# The export command creates the tflite model in the weights folder
try:
    tflite_path = "snapchef_vision/edge_model/weights/best_saved_model/best_int8.tflite"
    files.download(tflite_path)
except Exception as e:
    print(f"Could not auto-download. Please manually download the file from the left sidebar: {tflite_path}")